In [ ]:
from google.colab import drive

In [ ]:
drive.mount('\content\drive')

MessageError: ignored

In [ ]:
import torch
from DeepPhys import DeepPhys
import torch.nn as nn
from DeepPhys import Attention_mask


import os
import cv2
import torch
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image


import torch
import torch.nn as nn
import torch.optim as optim

In [ ]:
def load_pretrained(path):
  loaded_state_dict = torch.load(path,map_location=torch.device('cpu'))

  # In case the keys have the 'module.' prefix, remove it to match the keys in the current model
  if list(loaded_state_dict.keys())[0].startswith('module.'):
    loaded_state_dict = {k[7:]: v for k, v in loaded_state_dict.items()}

  return loaded_state_dict


In [ ]:
def current_model_create(in_channels, nb_filters1, nb_filters2, kernel_size,
                           dropout_rate1, dropout_rate2, pool_size, nb_dense, img_size):
    model = DeepPhys(
        in_channels=in_channels,
        nb_filters1=nb_filters1,
        nb_filters2=nb_filters2,
        kernel_size=kernel_size,
        dropout_rate1=dropout_rate1,
        dropout_rate2=dropout_rate2,
        pool_size=pool_size,
        nb_dense=nb_dense,
        img_size=img_size
    )
    return model



In [ ]:
def freeze_final_dense2_layer(model):
    for name, param in model.named_parameters():
        if not (name == 'final_dense_2.weight' or name == 'final_dense_2.bias'):
            param.requires_grad = False

In [ ]:
def check_layers_frozen(model):
    for name, param in model.named_parameters():
        print(f"Layer: {name}, Requires Grad: {param.requires_grad}")

In [ ]:
class VideoFrameDataset(Dataset):
    def __init__(self, root_folder, transform=None):
        self.root_folder = root_folder
        self.classes = os.listdir(root_folder)
        self.transform = transform

        '''
        up until here, we initialize the attributes of the VideoFramesDataset
        '''


        #creating two seperate lists for the video path and the labels
        self.video_paths = []
        self.labels = []

        '''
        below we iterate through the root folder. Each class is enumerated, i.e - given a
        value according to its index in the list self.classes . At the same time the name is
        stored in class name.
        '''

        for class_idx, class_name in enumerate(self.classes):
            class_folder = os.path.join(root_folder, class_name) # creating a path for the class folder present in the root folder
            video_files = os.listdir(class_folder) #video files contain all the video present in the class folder

            for video_file in video_files:
                video_path = os.path.join(class_folder, video_file) #constructing path for each video file
                self.video_paths.append(video_path) # adding the path to the self.video paths list
                self.labels.append(class_idx) # adding corresponding label value to the self.labels list


    def __len__(self):
        return len(self.video_paths)

    '''
    Below is the get item fucntion that is implicitly called by the video loaded.
    It returns the frames of a video along with the label.

    Frames is returned as a numpy array
    '''
    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]

        frames = self.extract_frames(video_path)
        if self.transform:
            # Convert each frame (NumPy array) to a PIL Image
            frames = [Image.fromarray(frame) for frame in frames]
            # Apply the transformation
            frames = [self.transform(frame) for frame in frames]

        return frames, label


    '''
    this function is to get the frames from the video specified by the video file path
    extract the frames from the video and return it.
    '''

    def extract_frames(self, video_path, target_frame_rate=30):
        frames = []

        cap = cv2.VideoCapture(video_path)

        # Get video properties
        frame_rate = int(cap.get(cv2.CAP_PROP_FPS))
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

        # Check for valid frame rate
        if frame_rate <= 0:
            print(f"Invalid frame rate ({frame_rate}). Check the video file or properties.")
            cap.release()
            return frames

        # Calculate the interval between frames to achieve the target frame rate
        interval = max(1, int(frame_rate / target_frame_rate))

        frame_count = 0
        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break
            if frame_count % interval == 0:
                frames.append(frame)
            frame_count += 1

        cap.release()

        return frames

In [ ]:
def train(model, train_loader, criterion, optimizer, num_epochs):
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        for inputs, labels in train_loader:
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        # Print training loss for this epoch
        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader)}")



In [ ]:
def validate(model, val_loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        val_accuracy = correct / total
        print(f"Validation Accuracy: {val_accuracy * 100}%")


In [ ]:
def main():
  path_pretrained="/content/UBFC-rPPG_DeepPhys.pth"

  loaded_state_dict=load_pretrained(path_pretrained)

  current_model = current_model_create(
    in_channels=3,
    nb_filters1=32,
    nb_filters2=64,
    kernel_size=3,
    dropout_rate1=0.25,
    dropout_rate2=0.5,
    pool_size=(2, 2),
    nb_dense=128,
    img_size=72)

  current_model.load_state_dict(loaded_state_dict)
  freeze_final_dense2_layer(current_model)
  check_layers_frozen(current_model)






In [ ]:
main()

In [ ]:
transform = transforms.Compose([
    transforms.Resize((72, 72)),
    transforms.ToTensor(),
])


In [ ]:
root_folder = '#path over here'
batch_size = 2

video_dataset = VideoFrameDataset(root_folder, transform=transform)
video_loader = torch.utils.data.DataLoader(video_dataset, batch_size=batch_size, shuffle=True)

In [ ]:
# Split your dataset into train and validation sets
train_size = int(0.8 * len(video_dataset))
val_size = len(video_dataset) - train_size
train_dataset, val_dataset = torch.utils.data.random_split(video_dataset, [train_size, val_size])


In [ ]:
batch_size = 2
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=batch_size)


In [ ]:
batch_size = 32
image_height = 72
image_width = 72
num_channels = 3


In [ ]:
criterion = nn.BCELoss()  # Binary Cross-Entropy Loss for binary classification
optimizer = optim.Adam(current_model.parameters(), lr=0.001)

In [ ]:
# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    for inputs, labels in train_loader:  # Iterate through the training loader
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
    # Print training loss for this epoch
    print(f"Epoch {epoch+1}/{num_epochs}, Training Loss: {running_loss/len(train_loader)}")

    # Validation loop (optional)
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:  # Iterate through the validation loader
            outputs = model(inputs)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
        val_accuracy = correct / total
        print(f"Validation Accuracy: {val_accuracy * 100}%")

In [ ]:
# Load the pre-trained model
weights_path = '/content/UBFC-rPPG_DeepPhys.pth'
state_dict = torch.load(weights_path, map_location=torch.device('cpu'))
state_dict



Below is the pretrained model with each of the weights and biases loaded as module.layer.weight or module.layer.bias


In [ ]:
# View the architecture of the pre-trained model
for key, value in state_dict.items():
    # if 'module.' in key:
        # key = key.replace('module.', '')
    print(key)

In [ ]:
state_dict

In [ ]:
from torchsummary import summary

In [ ]:
# View the architecture of DeepPhys
model = DeepPhys()
print(model)


DeepPhys(
  (motion_conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (motion_conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1))
  (motion_conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (motion_conv4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
  (apperance_conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (apperance_conv2): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1))
  (apperance_conv3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (apperance_conv4): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1))
  (apperance_att_conv1): Conv2d(32, 1, kernel_size=(1, 1), stride=(1, 1))
  (attn_mask_1): Attention_mask()
  (apperance_att_conv2): Conv2d(64, 1, kernel_size=(1, 1), stride=(1, 1))
  (attn_mask_2): Attention_mask()
  (avg_pooling_1): AvgPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0)
  (avg_pooling_2): AvgPool2d(kernel_size=(2, 2), stride=(2, 2), padding=0)
  (avg

at this point , we have modified the final layer to give a single output , indicated by the final dense layer having a sigmoid function in the forward method

below is the inputs of the model.

In [ ]:
# Initialize the model with the same parameters as the original model
current_model = DeepPhys(in_channels=3, nb_filters1=32, nb_filters2=64, kernel_size=3,
                         dropout_rate1=0.25, dropout_rate2=0.5, pool_size=(2, 2), nb_dense=128, img_size=72)


In [ ]:
loaded_state_dict = torch.load('/content/UBFC-rPPG_DeepPhys.pth',map_location=torch.device('cpu'))


Below we remove the prefix module from the state_dictionary

In [ ]:
# In case the keys have the 'module.' prefix, remove it to match the keys in the current model
if list(loaded_state_dict.keys())[0].startswith('module.'):
    loaded_state_dict = {k[7:]: v for k, v in loaded_state_dict.items()}


In [ ]:
# Load the matched state_dict into the model
current_model.load_state_dict(loaded_state_dict)


<All keys matched successfully>

Above shows that we have successfully mapped all weight and bias to the layers.

Note that we modified the DeepPhys class to match the sizes of the .pth file

Next steps should be the following
1. Learn to modify the architecture, adding a final dense layer. // but final dense layer already has final layer as single output.
2. Freeze all layers before it
3. Understand the input structure to the model.
4. Understand the input and output of the model, and how you would feed data, in the form of images to it , while taking into account the batch size and input shape and all

Below we try and freeze all but the final layer

In [ ]:
# Freeze only the final_dense_2 layer
freeze_final_dense2_layer(current_model)

In [ ]:
check_layers_frozen(current_model)

Layer: motion_conv1.weight, Requires Grad: False
Layer: motion_conv1.bias, Requires Grad: False
Layer: motion_conv2.weight, Requires Grad: False
Layer: motion_conv2.bias, Requires Grad: False
Layer: motion_conv3.weight, Requires Grad: False
Layer: motion_conv3.bias, Requires Grad: False
Layer: motion_conv4.weight, Requires Grad: False
Layer: motion_conv4.bias, Requires Grad: False
Layer: apperance_conv1.weight, Requires Grad: False
Layer: apperance_conv1.bias, Requires Grad: False
Layer: apperance_conv2.weight, Requires Grad: False
Layer: apperance_conv2.bias, Requires Grad: False
Layer: apperance_conv3.weight, Requires Grad: False
Layer: apperance_conv3.bias, Requires Grad: False
Layer: apperance_conv4.weight, Requires Grad: False
Layer: apperance_conv4.bias, Requires Grad: False
Layer: apperance_att_conv1.weight, Requires Grad: False
Layer: apperance_att_conv1.bias, Requires Grad: False
Layer: apperance_att_conv2.weight, Requires Grad: False
Layer: apperance_att_conv2.bias, Requires 

above, we can see that all but the final layer has the trainable parameter set to False, which means only the final dense's gradients will be computed during backprop

How to convert the deepfake videos to a form which is acceptable by the model.

Next step is to understand how to train this.
Given that we have a set of images in a folder , along with their labels, how would we pass it to the model.



Note that we have modified the DeepPhys Architecture such

---

that the, in the forward method of the architecture, we have made the final output a sigmoid value.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


In [ ]:
batch_size = 32
image_height = 72
image_width = 72
num_channels = 3

# Create random images and labels
random_images = torch.rand(batch_size, num_channels,image_height, image_width )
random_labels = torch.randint(2, (batch_size, 1), dtype=torch.float32)  # Random 0s and 1s as labels


In [ ]:
criterion = nn.BCELoss()  # Binary Cross-Entropy Loss for binary classification
optimizer = optim.Adam(current_model.parameters(), lr=0.001)

In [ ]:
num_epochs = 30

for epoch in range(num_epochs):
    current_model.train()

    # Generate random input images and labels
    random_images = torch.rand(batch_size, 3, 72, 72)
    labels = torch.randint(0, 2, (batch_size, 1), dtype=torch.float32)  # Generate random 0s and 1s

    # Zero the gradients
    optimizer.zero_grad()

    # Forward pass
    outputs = current_model(random_images)

    # Calculate loss
    loss = criterion(outputs, labels)

    # Backpropagation and optimization
    loss.backward()
    optimizer.step()

    # Print loss for monitoring
    print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')


Epoch [24/30], Loss: 0.7278
Epoch [25/30], Loss: 0.6743
Epoch [26/30], Loss: 0.6895


KeyboardInterrupt: ignored